# Notebook 5.0: Publication-Ready Results & Visualizations (2018–2022)

**Research Deliverables — Statistical Summaries & Figures 1–5**

This notebook:
1. Loads the scored sentiment dataset (`data/processed/sentiment_df.parquet`) and optional full-sample VADER cache (`sentiment_vader_full`).
2. Assigns the four 2018–2022 structural market regimes (`assign_regime`) and computes quarterly aggregates (`aggregate_quarterly`), Mann-Kendall monotonic trend tests (`mann_kendall_test`), and four-regime statistical comparisons (`regime_comparison`).
3. Prints and exports summary statistics tables to `data/final/tables/`.
4. Generates publication-ready Figures 1–5 at 300 DPI in `data/final/figures/`:
   - **Figure 1:** Sentiment composition across all 7 tech subreddits.
   - **Figure 2 (RQ2):** Negative sentiment evolution across the 4 2018–2022 market regimes.
   - **Figure 3 (RQ3a):** Weekly negative sentiment divergence across all 7 technical specializations.
   - **Figure 4 (RQ3b):** Structural shift in negative sentiment (`4_Layoffs & Tightening` vs. `3_Hiring Boom`) with 95% bootstrap CIs.
   - **Figure 5 (RQ1):** VADER vs. RoBERTa agreement matrix.

In [ ]:
import sys
from pathlib import Path

nb_dir = Path.cwd()
project_root = nb_dir.parent if nb_dir.name == 'notebooks' else nb_dir
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import yaml
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.modeling import set_seed
from src.modeling.cache import load_parquet_cache
from src.modeling.stats import (
    REGIME_ORDER,
    aggregate_quarterly,
    assign_regime,
    mann_kendall_test,
    regime_comparison,
)

config_path = project_root / 'config' / 'config.yaml'
with open(config_path, 'r', encoding='utf-8') as f:
    cfg = yaml.safe_load(f)

seed = int(cfg['modeling']['seed'])
set_seed(seed)

processed_dir = project_root / cfg['paths']['processed_data_dir']
cache_dir = project_root / cfg['modeling']['cache_dir']
final_dir = project_root / cfg['paths']['final_dir']
fig_dir = final_dir / 'figures'
tables_dir = final_dir / 'tables'
for d in (fig_dir, tables_dir):
    d.mkdir(parents=True, exist_ok=True)

# Publication plot styling
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['axes.edgecolor'] = '#333333'
plt.rcParams['axes.linewidth'] = 0.8

print(f'Results & visualization environment initialized (fig_dir={fig_dir}).')

## Step 1: Load Scored Datasets & Normalize UTC Timestamps
Load `data/processed/sentiment_df.parquet` (plus cached `sentiment_vader_full` if available) and assign the 4 2018–2022 regimes.

In [ ]:
parquet_path = processed_dir / 'sentiment_df.parquet'
df_sentiment = pd.read_parquet(parquet_path)
df_sentiment['created_utc_dt'] = pd.to_datetime(df_sentiment['created_utc_dt'], utc=True)

start_ts = pd.Timestamp(cfg['analysis']['start_date'], tz='UTC')
end_ts = pd.Timestamp(cfg['analysis']['end_date'], tz='UTC') + pd.Timedelta(days=1) - pd.Timedelta(seconds=1)
df_sentiment = (
    df_sentiment[(df_sentiment['created_utc_dt'] >= start_ts) & (df_sentiment['created_utc_dt'] <= end_ts)]
    .copy()
    .reset_index(drop=True)
)
df_sentiment['week'] = (
    df_sentiment['created_utc_dt']
    .dt.tz_convert(None)
    .dt.to_period('W')
    .dt.start_time
)
df_sentiment['regime'] = assign_regime(df_sentiment['created_utc_dt'])

vader_full = load_parquet_cache('sentiment_vader_full', cache_dir=cache_dir)
if vader_full is not None:
    vader_full['created_utc_dt'] = pd.to_datetime(vader_full['created_utc_dt'], utc=True)
    vader_full = (
        vader_full[(vader_full['created_utc_dt'] >= start_ts) & (vader_full['created_utc_dt'] <= end_ts)]
        .copy()
        .reset_index(drop=True)
    )

print(f'Loaded RoBERTa scored dataset: {len(df_sentiment):,} comments across {df_sentiment["subreddit"].nunique()} subreddits.')
if vader_full is not None:
    print(f'Loaded full VADER cross-check cache: {len(vader_full):,} comments.')
print('\nRegime counts (RoBERTa sample):')
print(df_sentiment['regime'].value_counts().reindex(REGIME_ORDER))

## Step 2: Summary Statistics Tables (`src/modeling/stats.py`)
Compute and print:
- Subreddit RoBERTa sentiment distribution
- Quarterly sentiment aggregates (`aggregate_quarterly`)
- Mann-Kendall monotonic trend tests (`mann_kendall_test`)
- Four-regime Kruskal-Wallis & Mann-Whitney comparisons with Holm-Bonferroni correction (`regime_comparison`)

In [ ]:
# 1. Subreddit sentiment distribution
sub_dist = (
    df_sentiment.groupby('subreddit')['roberta_label']
    .value_counts(normalize=True)
    .unstack(fill_value=0)
    * 100
).reindex(columns=['negative', 'neutral', 'positive'], fill_value=0.0).round(2)
sub_dist['n_roberta'] = df_sentiment.groupby('subreddit').size()
sub_dist.reset_index().to_csv(tables_dir / 'subreddit_sentiment_roberta.csv', index=False)
print('=== Table 1: RoBERTa Sentiment Distribution by Subreddit (%) ===')
print(sub_dist.to_string())

# 2. Quarterly aggregation
q_roberta = aggregate_quarterly(df_sentiment, label_col='roberta_label')
q_roberta.to_csv(tables_dir / 'quarterly_sentiment_roberta.csv', index=False)
if vader_full is not None:
    q_vader = aggregate_quarterly(vader_full, label_col='vader_label')
    q_vader.to_csv(tables_dir / 'quarterly_sentiment_vader_full.csv', index=False)
print('\n=== Table 2: Quarterly Overall Sentiment (RoBERTa, 2018Q1–2022Q4) ===')
print(q_roberta[q_roberta['subreddit'] == 'ALL'].sort_values('quarter').to_string(index=False))

# 3. Mann-Kendall quarterly monotonic trend tests
mk_rows = []
for sub, grp in q_roberta.groupby('subreddit', sort=True):
    grp_s = grp.sort_values('quarter')
    mk_neg = mann_kendall_test(grp_s['neg_pct'])
    mk_net = mann_kendall_test(grp_s['net_mean'])
    mk_rows.append({
        'subreddit': sub,
        'n_quarters': mk_neg['n'],
        'neg_pct_trend': mk_neg['trend'],
        'neg_pct_z': mk_neg['z'],
        'neg_pct_p': mk_neg['p_value'],
        'neg_pct_sen_slope_per_q': mk_neg['sen_slope'],
        'net_sent_trend': mk_net['trend'],
        'net_sent_p': mk_net['p_value'],
        'net_sent_sen_slope_per_q': mk_net['sen_slope'],
    })
mk_df = pd.DataFrame(mk_rows)
mk_df.to_csv(tables_dir / 'mann_kendall_trends.csv', index=False)
print('\n=== Table 3: Mann-Kendall Quarterly Trend Tests (RoBERTa) ===')
print(mk_df.to_string(index=False))

# 4. Four-regime statistical comparison
reg_roberta = regime_comparison(df_sentiment, label_col='roberta_label', n_boot=500, seed=seed)
reg_roberta.to_csv(tables_dir / 'regime_comparison_roberta.csv', index=False)
print('\n=== Table 4: Four-Regime Negative Sentiment Comparison (RoBERTa) ===')
print(reg_roberta.to_string(index=False))

if vader_full is not None:
    reg_vader = regime_comparison(vader_full, label_col='vader_label', n_boot=500, seed=seed)
    reg_vader.to_csv(tables_dir / 'regime_comparison_vader_full.csv', index=False)
    print('\n=== Table 5: Four-Regime Negative Sentiment Comparison (Full VADER) ===')
    print(reg_vader.to_string(index=False))

## Step 3: Figure 1 — Sentiment Composition by Subreddit
Stacked bar chart of RoBERTa sentiment across all 7 subreddits.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5.2))
sub_sentiment = df_sentiment.groupby(['subreddit', 'roberta_label']).size().unstack(fill_value=0)
sub_sentiment_pct = (
    sub_sentiment.div(sub_sentiment.sum(axis=1), axis=0) * 100
).reindex(columns=['negative', 'neutral', 'positive'], fill_value=0.0)

sub_sentiment_pct.plot(
    kind='bar',
    stacked=True,
    color=['#d9534f', '#f0ad4e', '#5cb85c'],
    ax=ax,
    edgecolor='none',
    width=0.6,
)

ax.set_title('Figure 1: RoBERTa Sentiment Distribution by Subreddit Community (2018–2022)', fontsize=13, pad=12)
ax.set_xlabel('Subreddit Cohort', fontsize=11)
ax.set_ylabel('Percentage of Comments (%)', fontsize=11)
ax.legend(title='RoBERTa Sentiment', bbox_to_anchor=(1.02, 1), loc='upper left')
plt.xticks(rotation=25, ha='right')
plt.tight_layout()

fig1_path = fig_dir / 'fig1_sentiment_by_subreddit.png'
plt.savefig(fig1_path, dpi=300)
plt.show()
print(f'Saved: {fig1_path}')

## Step 4: Figure 2 (RQ2) — Negative Sentiment Evolution Across 2018–2022 Regimes
Four-week rolling negative sentiment share overlaid on the 4 structural regimes (`1_Pre-COVID`, `2_COVID Freeze`, `3_Hiring Boom`, `4_Layoffs & Tightening`).

In [ ]:
fig, ax = plt.subplots(figsize=(12.5, 5.2))

weekly_neg = (
    df_sentiment.groupby(['week', 'roberta_label'])
    .size()
    .unstack(fill_value=0)
    .reindex(columns=['negative', 'neutral', 'positive'], fill_value=0)
)
weekly_neg_share = (weekly_neg['negative'] / weekly_neg.sum(axis=1)) * 100

ax.plot(
    weekly_neg_share.index,
    weekly_neg_share.rolling(4, min_periods=1).mean(),
    color='#c9302c',
    linewidth=2.3,
    label='RoBERTa Negative Share (4-week rolling avg)',
)

# Four 2018-2022 regimes matching src/modeling/stats.py
ax.axvspan(pd.Timestamp('2018-01-01'), pd.Timestamp('2020-02-29'), color='#bbbbbb', alpha=0.22, label='1. Pre-COVID (2018–2020Q1)')
ax.axvspan(pd.Timestamp('2020-03-01'), pd.Timestamp('2020-12-31'), color='#f0ad4e', alpha=0.18, label='2. COVID Freeze (2020)')
ax.axvspan(pd.Timestamp('2021-01-01'), pd.Timestamp('2022-05-31'), color='#5cb85c', alpha=0.15, label='3. Hiring Boom (2021–2022H1)')
ax.axvspan(pd.Timestamp('2022-06-01'), pd.Timestamp('2022-12-31'), color='#d9534f', alpha=0.18, label='4. Layoffs & Tightening (2022H2)')

ax.axvline(pd.Timestamp('2022-06-01'), color='#a94442', linestyle='--', linewidth=1.6, label='2022 Layoffs Onset (Jun 1, 2022)')
ax.axvline(pd.Timestamp('2022-11-30'), color='#000000', linestyle=':', linewidth=1.8, label='ChatGPT Launch (Nov 30, 2022)')

ax.set_title('Figure 2 (RQ2): Negative Sentiment Evolution Across 2018–2022 Macroeconomic Regimes', fontsize=13, pad=12)
ax.set_xlabel('Timeline (2018–2022)', fontsize=11)
ax.set_ylabel('Negative Sentiment Share (%)', fontsize=11)
ax.legend(bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=9)
ax.grid(True, linestyle='--', alpha=0.4)
plt.tight_layout()

fig2_path = fig_dir / 'fig2_regime_sentiment_evolution.png'
plt.savefig(fig2_path, dpi=300)
plt.show()
print(f'Saved: {fig2_path}')

## Step 5: Figure 3 (RQ3a) — Specialization Sentiment Divergence
Compare 4-week rolling negative sentiment share across all 7 technical subreddits (2018–2022).

In [ ]:
fig, ax = plt.subplots(figsize=(13, 6))

for spec in cfg['analysis']['subreddits']:
    spec_data = df_sentiment[df_sentiment['subreddit'] == spec]
    if len(spec_data) > 0:
        weekly = (
            spec_data.groupby(['week', 'roberta_label'])
            .size()
            .unstack(fill_value=0)
            .reindex(columns=['negative', 'neutral', 'positive'], fill_value=0)
        )
        neg_pct = (weekly['negative'] / weekly.sum(axis=1)) * 100
        ax.plot(neg_pct.index, neg_pct.rolling(4, min_periods=1).mean(), label=f'r/{spec}', linewidth=1.7, alpha=0.85)

ax.axvline(pd.Timestamp('2020-03-01'), color='#555555', linestyle='--', linewidth=1.4, label='COVID Freeze (Mar 2020)')
ax.axvline(pd.Timestamp('2022-06-01'), color='#d9534f', linestyle='--', linewidth=1.5, label='2022 Layoffs Onset (Jun 2022)')
ax.axvline(pd.Timestamp('2022-11-30'), color='#000000', linestyle=':', linewidth=1.7, label='ChatGPT Launch (Nov 30, 2022)')

ax.set_title('Figure 3 (RQ3a): Weekly Negative Sentiment Divergence Across 7 Tech Subreddits (2018–2022)', fontsize=13, pad=12)
ax.set_xlabel('Date (Weekly)', fontsize=11)
ax.set_ylabel('Negative Sentiment Share (%, 4-wk rolling)', fontsize=11)
ax.legend(bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=9)
ax.grid(True, linestyle='--', alpha=0.4)
plt.tight_layout()

fig3_path = fig_dir / 'fig3_specialization_sentiment_divergence.png'
plt.savefig(fig3_path, dpi=300)
plt.show()
print(f'Saved: {fig3_path}')

## Step 6: Figure 4 (RQ3b) — Structural Regime Shift (`2022H2 Layoffs` vs. `2021–2022H1 Hiring Boom`)
Visualize the percentage-point shift in negative sentiment share (`delta_layoffs_vs_boom_pp`) with 95% bootstrap confidence intervals, alongside a timezone-safe UTC break check.

In [ ]:
# Timezone-safe UTC structural break check
layoffs_break_utc = pd.Timestamp('2022-06-01', tz='UTC')
df_sentiment['post_layoffs_break'] = df_sentiment['created_utc_dt'] >= layoffs_break_utc
df_sentiment['is_negative'] = (df_sentiment['roberta_label'] == 'negative').astype(float) * 100.0

its_summary = df_sentiment.groupby(['subreddit', 'post_layoffs_break'])['is_negative'].mean().unstack()
its_summary.columns = ['Pre-2022H2 (%)', 'Layoffs 2022H2 (%)']
its_summary['Delta vs Pre-2022H2 (pp)'] = its_summary['Layoffs 2022H2 (%)'] - its_summary['Pre-2022H2 (%)']
its_summary.to_csv(tables_dir / 'its_sentiment_summary.csv')

print('=== Pre- vs. Post-2022-06-01 UTC Shift in Negative Sentiment (%) ===')
print(its_summary.round(2).to_string())

# Plot regime comparison delta (Layoffs 2022H2 vs Hiring Boom 2021-2022H1) with 95% bootstrap CIs
plot_reg = reg_roberta[reg_roberta['subreddit'] != 'ALL'].sort_values('delta_layoffs_vs_boom_pp')
y_pos = np.arange(len(plot_reg))
deltas = plot_reg['delta_layoffs_vs_boom_pp'].to_numpy(dtype=float)
err_low = np.maximum(0.0, deltas - plot_reg['ci_95_low'].to_numpy(dtype=float))
err_high = np.maximum(0.0, plot_reg['ci_95_high'].to_numpy(dtype=float) - deltas)
colors = ['#d9534f' if d >= 0 else '#5cb85c' for d in deltas]

fig, ax = plt.subplots(figsize=(9, 5))
ax.barh(y_pos, deltas, xerr=[err_low, err_high], color=colors, height=0.58, capsize=4, alpha=0.9)
ax.axvline(0, color='black', linestyle='-', linewidth=0.9)
ax.set_yticks(y_pos)
ax.set_yticklabels([f'r/{s}' for s in plot_reg['subreddit']])
ax.set_title('Figure 4 (RQ3b): Negative Sentiment Shift (2022H2 Layoffs vs. Hiring Boom, 95% CI)', fontsize=12, pad=10)
ax.set_xlabel('Change in Negative Sentiment Rate (Percentage Points)', fontsize=10.5)
ax.set_ylabel('Subreddit Cohort', fontsize=10.5)
ax.grid(True, axis='x', linestyle='--', alpha=0.4)
plt.tight_layout()

fig4_path = fig_dir / 'fig4_its_sentiment_delta.png'
plt.savefig(fig4_path, dpi=300)
plt.show()
print(f'Saved: {fig4_path}')

## Step 7: Figure 5 (RQ1) — VADER vs. RoBERTa Agreement Matrix
Normalized confusion matrix comparing baseline VADER and primary RoBERTa classifications.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))
labels_order = ['negative', 'neutral', 'positive']
confusion = pd.crosstab(df_sentiment['vader_label'], df_sentiment['roberta_label'], normalize='all') * 100
confusion = confusion.reindex(index=labels_order, columns=labels_order, fill_value=0.0)

sns.heatmap(
    confusion,
    annot=True,
    fmt='.1f',
    cmap='Blues',
    cbar=False,
    ax=ax,
    linewidths=1,
)

agreement = (df_sentiment['vader_label'] == df_sentiment['roberta_label']).mean()
ax.set_title(f'Figure 5 (RQ1): VADER vs. RoBERTa Agreement Matrix (% of N={len(df_sentiment):,}, Agreement={agreement:.1%})', fontsize=11.5, pad=12)
ax.set_xlabel('RoBERTa Classification (Primary)', fontsize=11)
ax.set_ylabel('VADER Classification (Baseline)', fontsize=11)
plt.tight_layout()

fig5_path = fig_dir / 'fig5_vader_roberta_agreement_matrix.png'
plt.savefig(fig5_path, dpi=300)
plt.show()
print(f'Saved: {fig5_path}')
print('All 5 publication figures and summary tables generated successfully.')